# Retrieval Validation — run + LLM-judge the shipped pipeline for any gene

Runs the **current shipped** literature-retrieval pipeline for a single gene and scores the
final papers with an LLM-as-judge. No cross-encoder / LLM-judgment stage in the live pipeline
yet — the scoring here is a standalone evaluation layer (`reactome_llm/retrieval_eval.py`).

**Pipeline recap (what this notebook exercises):**

| Branch | Stage-1 `name_query` (≤100) | Stage-1 `context_query` (≤200) | Stage-2 re-rank target |
|---|---|---|---|
| Has-data | gene + synonyms | direct pathway names **+ partners** | real pathway summaries (max-cosine) |
| Cold-start, gate-pass | gene + synonyms | enriched pathway names **+ partners** | primary suggested pathway summary |
| Cold-start, gate-fail | gene + synonyms | *(empty)* | LLM gene description (DESC) |

Stage 1 runs the two queries as **separate** E-Searches and unions/dedups by PMID (~300 pool,
smaller for gate-fail). Stage 2 is the only ranking stage: bi-encoder cosine similarity, keep
top `MAX_PAPERS` (5) directly — no intermediate cut. Partners live in `context_query` only, so
the gate-fail query is gene + synonyms (no partners).

## 0. Setup

In [ ]:
import os, sys
# Operate from repo root so resource paths resolve, whether launched from notebooks/ or root.
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.append('reactome_llm')
sys.path.append('.')

from retrieval_eval import run_retrieval, score_retrieval_relevance
from QueryBuilder import build_query_and_search_terms
import pandas as pd

## 1. Pick a gene

Try `TANC1` (cold-start, gate-fail), `BRCA1` / `TP53` (has-data), or any gene symbol.

In [ ]:
GENE = 'TANC1'

## 2. Run the shipped pipeline

Determines the branch, builds the query pair, runs the E-Search merge, and re-ranks the pool
against the branch-appropriate Stage-2 target — via the real `LiteratureSearchTool` (so this
cannot drift from production). For gate-fail genes it replicates the annotator's upfront DESC
precompute exactly as the live pipeline does.

In [ ]:
run = run_retrieval(GENE)

print(f"Gene          : {run['gene']}")
print(f"Branch        : {run['branch']}")
print(f"name_query    : {run['name_query']}")
print(f"context_query : {run['context_query'] or '(empty — gate-fail, name-only pool)'}")
print(f"Stage-1 pool  : {run['pool_size']} papers")
print(f"Stage-2 target: {len(run['target_text'])} text(s), max-cosine across them")
print(f"  first target: {(run['target_text'][0] if run['target_text'] else '')[:300]}")

## 3. LLM-as-judge: curator-style annotation usefulness

One **batched** LLM call scores each final paper as an expert Reactome curator would — for
**usefulness in building an annotation**, not mere topical relevance. Per paper it reasons
through: (1) does it describe a *specific* molecular interaction/reaction involving the gene
(or its immediate partner/pathway context)? (2) evidence type — **DIRECT** (co-IP, in vitro
binding, crystallography, SPR/ITC), **INDIRECT** (knockout, overexpression, inhibitor), or
**INSUFFICIENT** (microarray, bulk proteomics, computational alone); (3) is it strong enough
to plausibly support a Reactome entity/reaction/complex, or just background? The gene
description (gate-fail DESC when available, else generated here) is passed as reference context.

In [ ]:
judge_description = run['description'] or build_query_and_search_terms(GENE)[1]
scored = score_retrieval_relevance(GENE, judge_description, run['papers'])

print(f"mean usefulness   : {scored['mean_score']}")
print(f"score distribution: {scored['distribution']}   (score: count)")
print(f"evidence types    : {scored['evidence_type_counts']}")
print(f"annotatable       : {scored['annotatable_count']}/{scored['n']}")

## 4. Final result — the 5 papers with curator scores

In [ ]:
rows = []
for rank, (p, s) in enumerate(zip(run['papers'], scored['per_paper']), 1):
    snippet = ' '.join((p.get('abstract') or '').split())[:180]
    rows.append({
        'rank': rank,
        'pmid': p.get('pmid', ''),
        'title': p.get('title', '') or '(no title)',
        'score': s['score'],
        'evidence_type': s['evidence_type'],
        'annotatable': s['annotatable'],
        'justification': s['justification'],
        'abstract_snippet': snippet + '...',
    })

pd.set_option('display.max_colwidth', 80)
pd.DataFrame(rows).set_index('rank')